# AI Academic Advisor: Model Training
CSC 303 assignment. Builds a Random Forest classifier that predicts a student's academic standing (**0 = At Risk, 1 = Average, 2 = Excellent**).

> The dataset below is **synthetic**, generated for demonstration purposes.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 2. Generate the training data
The `standing` label comes from a weighted score of the features plus noise, split into three equal groups.

In [ ]:
np.random.seed(42)
n = 1500

df = pd.DataFrame({
    "prev_cgpa": np.round(np.random.uniform(1.0, 5.0, n), 2),
    "study_hours": np.random.randint(0, 40, n),
    "attendance": np.random.randint(40, 101, n),
    "courses_failed": np.random.randint(0, 5, n),
    "sleep_hours": np.random.randint(3, 10, n),
})

score = (0.6 * df.prev_cgpa + 0.04 * df.study_hours + 0.02 * df.attendance
         - 0.3 * df.courses_failed + 0.05 * df.sleep_hours
         + np.random.normal(0, 0.3, n))

df["standing"] = pd.qcut(score, 3, labels=[0, 1, 2]).astype(int)
df.head(10)

## 3. Explore the data

In [ ]:
print(df.shape)
print(df['standing'].value_counts().sort_index())
df.describe()

In [ ]:
df.hist(figsize=(12, 7), bins=20)
plt.tight_layout()
plt.show()

## 4. Save the training data as CSV

In [ ]:
df.to_csv('student_data.csv', index=False)
print('Saved student_data.csv')

## 5. Train / test split

In [ ]:
X = df.drop('standing', axis=1)
y = df['standing']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

## 6. Train the Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

## 7. Evaluate

In [ ]:
pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, target_names=['At Risk', 'Average', 'Excellent']))
print(confusion_matrix(y_test, pred))

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importances.plot(kind='barh', title='Feature importance')
plt.show()

## 8. Test a single prediction

In [ ]:
sample = pd.DataFrame([{'prev_cgpa': 3.8, 'study_hours': 25, 'attendance': 90,
                        'courses_failed': 0, 'sleep_hours': 7}])
labels = {0: 'At Risk', 1: 'Average', 2: 'Excellent'}
print(labels[int(model.predict(sample)[0])])

## 9. Export the model (.pkl)
Download `advisor_model.pkl` and place it in the project's `model/` folder.

In [ ]:
joblib.dump(model, 'advisor_model.pkl')

try:
    from google.colab import files
    files.download('advisor_model.pkl')
    files.download('student_data.csv')
except ImportError:
    print('Not running in Colab: files saved in the current folder.')